# Evaluation 2 — Gradient Boosting Development

This notebook is **Reshmi's primary algorithm contribution**. It develops `GradientBoostingClassifier` on the approved outer-training population only: a shared baseline, controlled grouped-CV tuning for General / City / Resort scopes, grouped out-of-fold (OOF) evidence, and frozen configurations for later group-level outer-test evaluation.

**NO OUTER-TEST MODEL PERFORMANCE IS CALCULATED IN THIS NOTEBOOK.**

The approved outer split is reconstructed only to recover the exact outer-training rows. Outer-test labels and predictions are never used for tuning, selection, feature decisions, threshold tuning, or specialist-strategy selection.

## Why Gradient Boosting?

Gradient Boosting builds trees sequentially. Each new weak learner attempts to correct errors remaining from the previous ensemble. This lets it represent nonlinear relationships, interactions among booking characteristics, and complex cancellation patterns.

Strengths include strong performance on structured/tabular classification, nonlinear effects, the combination of many relatively weak trees, probability estimates, and a useful comparison with Logistic Regression, Decision Tree, and Random Forest. Limitations are that sequential training is slower than one tree, aggressive learning or overly complex trees may overfit, learning rate and estimator count interact, and the ensemble is less directly interpretable than Logistic Regression or one Decision Tree. These are reasons to evaluate it—not a claim that it is the best algorithm.

## 1. Imports and fixed experiment contract

The shared project utilities remain authoritative. Preprocessing stays inside every model pipeline so imputation, scaling, and one-hot vocabularies are learned independently within each training fold.

In [1]:
from pathlib import Path
import json
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold, cross_validate

root_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    root for root in root_candidates
    if (root / 'src' / 'preprocessing.py').exists() and (root / 'data' / 'raw').exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import (
    TARGET, create_predictor_groups, prepare_training_table, split_features_target,
)
from src.modeling import (
    RANDOM_STATE, PRIMARY_SCORING, COMMON_SCORING, MODEL_SCOPES,
    make_inner_cv, create_training_scopes, summarize_training_scopes,
    build_model_pipeline, CV_RESULT_COLUMNS,
)

RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'hotel_bookings.csv'
REPORT_DIR = PROJECT_ROOT / 'reports' / 'eval2'
CV_RESULTS_PATH = REPORT_DIR / 'gradient_boosting_cv_results.csv'
OOF_RESULTS_PATH = REPORT_DIR / 'gradient_boosting_oof_results.csv'
ALGORITHM = 'Gradient Boosting'
CV_STRATEGY = 'StratifiedGroupKFold(5)'
N_JOBS = 2
assert RANDOM_STATE == 42
assert PRIMARY_SCORING == 'f1'
assert tuple(MODEL_SCOPES) == ('General', 'City', 'Resort')
assert set(COMMON_SCORING) == {'accuracy', 'precision', 'recall', 'f1', 'roc_auc'}

## 2. Recover the approved outer-training population

The first split of the approved 5-fold `StratifiedGroupKFold` is recreated exactly. Only outer-training objects are retained for development; the test positions are counted only to verify the approved partition size.

In [2]:
raw_df = pd.read_csv(RAW_PATH)
training_table = prepare_training_table(raw_df)
X, y = split_features_target(training_table)
predictor_groups = create_predictor_groups(X)

outer_splitter = StratifiedGroupKFold(
    n_splits=5, shuffle=True, random_state=RANDOM_STATE
)
train_positions, test_positions = next(
    outer_splitter.split(X, y, groups=predictor_groups)
)
assert len(X) == 119_210
assert len(train_positions) == 95_375
assert len(test_positions) == 23_835
assert set(train_positions).isdisjoint(test_positions)

X_train_outer = X.iloc[train_positions].copy()
y_train_outer = y.iloc[train_positions].copy()
groups_train_outer = predictor_groups.iloc[train_positions].copy()

del raw_df, training_table, X, y, predictor_groups, outer_splitter, train_positions, test_positions
print('Approved outer-training population recovered; outer-test performance remains untouched.')

Approved outer-training population recovered; outer-test performance remains untouched.


## 3. Create the approved model scopes

The specialist scopes are subsets of the same outer-training population. They retain `hotel` and the full common 30-column predictor schema; no separate random split is created.

In [3]:
training_scopes = create_training_scopes(
    X_train_outer, y_train_outer, groups_train_outer
)
scope_summary = summarize_training_scopes(training_scopes)
expected_scope_rows = {'General': 95_375, 'City': 63_044, 'Resort': 32_331}
assert {name: len(training_scopes[name]['X']) for name in MODEL_SCOPES} == expected_scope_rows
assert all(training_scopes[name]['X'].shape[1] == 30 for name in MODEL_SCOPES)
assert all('hotel' in training_scopes[name]['X'].columns for name in MODEL_SCOPES)
display(scope_summary)

,scope,rows,cancellation_rate,unique_predictor_groups
0,General,95375,0.370779,66831
1,City,63044,0.416915,40872
2,Resort,32331,0.280814,25959


## 4. Fixed baseline

All scopes use the same baseline. `n_estimators=100` is a moderate ensemble size; `learning_rate=0.1` is a common moderate boosting step; `max_depth=3` makes shallow weak learners; `subsample=0.8` gives stochastic boosting; `max_features='sqrt'` reduces split-search computation and introduces diversity; and `random_state=42` makes the stochastic choices reproducible.

In [4]:
BASELINE_PARAMETERS = {
    'n_estimators': 100,
    'learning_rate': 0.1,
    'max_depth': 3,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'subsample': 0.8,
    'max_features': 'sqrt',
    'random_state': RANDOM_STATE,
}

def json_parameters(parameters):
    return json.dumps(parameters, sort_keys=True)

def cv_result_record(scope_name, stage, parameters, scores, tuning_method=None, best_parameters=None):
    scope = training_scopes[scope_name]
    record = {
        'algorithm': ALGORITHM,
        'scope': scope_name,
        'stage': stage,
        'cv_strategy': CV_STRATEGY,
        'random_state': RANDOM_STATE,
        'parameters': json_parameters(parameters),
        'training_rows': len(scope['X']),
        'predictor_groups': scope['groups'].nunique(),
        'tuning_method': tuning_method,
        'best_parameters': None if best_parameters is None else json_parameters(best_parameters),
        'observation': 'Grouped outer-training CV; no outer-test metric.',
    }
    for metric in ('accuracy', 'precision', 'recall', 'f1', 'roc_auc'):
        values = np.asarray(scores[metric], dtype=float)
        record[f'mean_cv_{metric}'] = values.mean()
        record[f'std_cv_{metric}'] = values.std()
    assert set(record) == set(CV_RESULT_COLUMNS)
    return record

## 5. Baseline grouped cross-validation

Each complete preprocessing/model pipeline is cloned within grouped CV. The following cell performs the baseline computation only when the student runs this notebook locally.

In [5]:
baseline_records = []
for scope_name in MODEL_SCOPES:
    scope = training_scopes[scope_name]
    baseline_estimator = GradientBoostingClassifier(**BASELINE_PARAMETERS)
    baseline_pipeline = build_model_pipeline(scope['X'], baseline_estimator)
    baseline_cv = cross_validate(
        baseline_pipeline,
        scope['X'],
        scope['y'],
        groups=scope['groups'],
        cv=make_inner_cv(),
        scoring=COMMON_SCORING,
        return_train_score=False,
        n_jobs=N_JOBS,
    )
    baseline_scores = {
        metric: baseline_cv[f'test_{metric}']
        for metric in ('accuracy', 'precision', 'recall', 'f1', 'roc_auc')
    }
    baseline_records.append(
        cv_result_record(
            scope_name, 'baseline', BASELINE_PARAMETERS, baseline_scores,
            tuning_method=None, best_parameters=None,
        )
    )

baseline_results = pd.DataFrame(baseline_records, columns=CV_RESULT_COLUMNS)
display(baseline_results)

,algorithm,scope,stage,cv_strategy,random_state,parameters,training_rows,predictor_groups,mean_cv_accuracy,std_cv_accuracy,...,std_cv_precision,mean_cv_recall,std_cv_recall,mean_cv_f1,std_cv_f1,mean_cv_roc_auc,std_cv_roc_auc,tuning_method,best_parameters,observation
0,Gradient Boosting,General,baseline,StratifiedGroupKFold(5),42,"{""learning_rate"": 0.1, ""max_depth"": 3, ""max_fe...",95375,66831,0.814480,0.002038,...,0.015193,0.581630,0.017801,0.699085,0.008401,0.910866,0.000914,None,None,Grouped outer-training CV; no outer-test metric.
1,Gradient Boosting,City,baseline,StratifiedGroupKFold(5),42,"{""learning_rate"": 0.1, ""max_depth"": 3, ""max_fe...",63044,40872,0.821372,0.007006,...,0.011494,0.649331,0.011598,0.751897,0.010248,0.909220,0.005129,None,None,Grouped outer-training CV; no outer-test metric.
2,Gradient Boosting,Resort,baseline,StratifiedGroupKFold(5),42,"{""learning_rate"": 0.1, ""max_depth"": 3, ""max_fe...",32331,25959,0.846185,0.006705,...,0.012301,0.558652,0.028001,0.670646,0.020473,0.924893,0.005879,None,None,Grouped outer-training CV; no outer-test metric.


## 6. Controlled randomized search

The space contains exactly $2^5=32$ theoretical combinations. Only four candidates are sampled per scope: 4 candidates × 5 grouped folds = 20 tuning fits per scope and 60 tuning fits across all three scopes. F1 is the refit and selection metric. No extra tuning stage, feature-selection experiment, or decision-threshold tuning is performed.

In [6]:
PARAM_DISTRIBUTIONS = {
    'classifier__n_estimators': [50, 100],
    'classifier__learning_rate': [0.05, 0.1],
    'classifier__max_depth': [2, 3],
    'classifier__min_samples_leaf': [1, 10],
    'classifier__subsample': [0.8, 1.0],
}
N_SEARCH_CANDIDATES = 4
THEORETICAL_COMBINATIONS = int(np.prod([len(values) for values in PARAM_DISTRIBUTIONS.values()]))
TUNING_FITS_PER_SCOPE = N_SEARCH_CANDIDATES * 5
TOTAL_TUNING_FITS = TUNING_FITS_PER_SCOPE * len(MODEL_SCOPES)
assert THEORETICAL_COMBINATIONS == 32
assert N_SEARCH_CANDIDATES == 4
assert TUNING_FITS_PER_SCOPE == 20
assert TOTAL_TUNING_FITS == 60
assert set(PARAM_DISTRIBUTIONS) == {
    'classifier__n_estimators', 'classifier__learning_rate',
    'classifier__max_depth', 'classifier__min_samples_leaf',
    'classifier__subsample',
}

In [7]:
tuned_records = []
selected_pipeline_parameters = {}
selected_estimator_parameters = {}
searches = {}

for scope_name in MODEL_SCOPES:
    scope = training_scopes[scope_name]
    fixed_search_estimator = GradientBoostingClassifier(
        max_features='sqrt', min_samples_split=2, random_state=RANDOM_STATE
    )
    search_pipeline = build_model_pipeline(scope['X'], fixed_search_estimator)
    search = RandomizedSearchCV(
        estimator=search_pipeline,
        param_distributions=PARAM_DISTRIBUTIONS,
        n_iter=N_SEARCH_CANDIDATES,
        scoring=COMMON_SCORING,
        refit=PRIMARY_SCORING,
        cv=make_inner_cv(),
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
        return_train_score=False,
    )
    search.fit(scope['X'], scope['y'], groups=scope['groups'])
    assert len(search.cv_results_['params']) == N_SEARCH_CANDIDATES == 4
    assert search.refit == PRIMARY_SCORING == 'f1'

    best_index = search.best_index_
    assert np.isclose(search.best_score_, search.cv_results_['mean_test_f1'][best_index])
    best_pipeline_params = dict(search.best_params_)
    best_estimator_params = {
        key.removeprefix('classifier__'): value
        for key, value in best_pipeline_params.items()
    }
    frozen_params = {
        'max_features': 'sqrt',
        'min_samples_split': 2,
        'random_state': RANDOM_STATE,
        **best_estimator_params,
    }
    selected_pipeline_parameters[scope_name] = best_pipeline_params
    selected_estimator_parameters[scope_name] = frozen_params
    searches[scope_name] = search

    best_scores = {
        metric: np.array([search.cv_results_[f'mean_test_{metric}'][best_index]])
        for metric in ('accuracy', 'precision', 'recall', 'f1', 'roc_auc')
    }
    tuned_record = cv_result_record(
        scope_name, 'tuned', frozen_params, best_scores,
        tuning_method='RandomizedSearchCV',
        best_parameters=best_pipeline_params,
    )
    for metric in ('accuracy', 'precision', 'recall', 'f1', 'roc_auc'):
        tuned_record[f'mean_cv_{metric}'] = search.cv_results_[f'mean_test_{metric}'][best_index]
        tuned_record[f'std_cv_{metric}'] = search.cv_results_[f'std_test_{metric}'][best_index]
    tuned_records.append(tuned_record)

tuned_results = pd.DataFrame(tuned_records, columns=CV_RESULT_COLUMNS)
display(tuned_results)

,algorithm,scope,stage,cv_strategy,random_state,parameters,training_rows,predictor_groups,mean_cv_accuracy,std_cv_accuracy,...,std_cv_precision,mean_cv_recall,std_cv_recall,mean_cv_f1,std_cv_f1,mean_cv_roc_auc,std_cv_roc_auc,tuning_method,best_parameters,observation
0,Gradient Boosting,General,tuned,StratifiedGroupKFold(5),42,"{""learning_rate"": 0.1, ""max_depth"": 3, ""max_fe...",95375,66831,0.777335,0.003629,...,0.015783,0.420064,0.017645,0.582907,0.013920,0.896392,0.004858,RandomizedSearchCV,"{""classifier__learning_rate"": 0.1, ""classifier...",Grouped outer-training CV; no outer-test metric.
1,Gradient Boosting,City,tuned,StratifiedGroupKFold(5),42,"{""learning_rate"": 0.1, ""max_depth"": 3, ""max_fe...",63044,40872,0.788644,0.011798,...,0.028639,0.520558,0.042533,0.671447,0.029833,0.899786,0.007222,RandomizedSearchCV,"{""classifier__learning_rate"": 0.1, ""classifier...",Grouped outer-training CV; no outer-test metric.
2,Gradient Boosting,Resort,tuned,StratifiedGroupKFold(5),42,"{""learning_rate"": 0.1, ""max_depth"": 3, ""max_fe...",32331,25959,0.810770,0.011205,...,0.018141,0.381434,0.053354,0.528579,0.051623,0.907421,0.010081,RandomizedSearchCV,"{""classifier__learning_rate"": 0.1, ""classifier...",Grouped outer-training CV; no outer-test metric.


## 7. Baseline versus tuned grouped-CV performance

Changes are descriptive training-only grouped-CV evidence. A small positive change is reported as small; a negative change means this limited randomized search did not improve the metric. No statistical significance is claimed.

In [8]:
comparison_rows = []
for scope_name in MODEL_SCOPES:
    baseline_row = baseline_results.loc[baseline_results['scope'].eq(scope_name)].iloc[0]
    tuned_row = tuned_results.loc[tuned_results['scope'].eq(scope_name)].iloc[0]
    comparison_rows.append({
        'scope': scope_name,
        'baseline F1': baseline_row['mean_cv_f1'],
        'tuned F1': tuned_row['mean_cv_f1'],
        'F1 change': tuned_row['mean_cv_f1'] - baseline_row['mean_cv_f1'],
        'baseline Recall': baseline_row['mean_cv_recall'],
        'tuned Recall': tuned_row['mean_cv_recall'],
        'Recall change': tuned_row['mean_cv_recall'] - baseline_row['mean_cv_recall'],
        'baseline ROC-AUC': baseline_row['mean_cv_roc_auc'],
        'tuned ROC-AUC': tuned_row['mean_cv_roc_auc'],
        'ROC-AUC change': tuned_row['mean_cv_roc_auc'] - baseline_row['mean_cv_roc_auc'],
    })
baseline_vs_tuned = pd.DataFrame(comparison_rows)
display(baseline_vs_tuned)

for row in comparison_rows:
    delta = row['F1 change']
    if delta < 0:
        interpretation = 'the limited randomized search did not improve F1'
    elif delta < 0.01:
        interpretation = 'the F1 improvement is small'
    else:
        interpretation = 'the tuned configuration improved grouped-CV F1'
    print(f"{row['scope']}: {interpretation} (change={delta:+.4f}).")

,scope,baseline F1,tuned F1,F1 change,baseline Recall,tuned Recall,Recall change,baseline ROC-AUC,tuned ROC-AUC,ROC-AUC change
0,General,0.699085,0.582907,-0.116178,0.581630,0.420064,-0.161567,0.910866,0.896392,-0.014473
1,City,0.751897,0.671447,-0.080450,0.649331,0.520558,-0.128772,0.909220,0.899786,-0.009433
2,Resort,0.670646,0.528579,-0.142068,0.558652,0.381434,-0.177218,0.924893,0.907421,-0.017472


General: the limited randomized search did not improve F1 (change=-0.1162).
City: the limited randomized search did not improve F1 (change=-0.0804).
Resort: the limited randomized search did not improve F1 (change=-0.1421).


## 8. Save the six grouped-CV records

This output is generated only when the notebook is run locally. It contains three baseline and three tuned rows in the shared schema.

In [9]:
cv_results = pd.concat([baseline_results, tuned_results], ignore_index=True)
cv_results = cv_results.loc[:, list(CV_RESULT_COLUMNS)]
assert list(cv_results.columns) == list(CV_RESULT_COLUMNS)
assert len(cv_results) == 6
assert cv_results['stage'].value_counts().to_dict() == {'baseline': 3, 'tuned': 3}
assert set(cv_results['scope']) == set(MODEL_SCOPES) == {'General', 'City', 'Resort'}
REPORT_DIR.mkdir(parents=True, exist_ok=True)
cv_results.to_csv(CV_RESULTS_PATH, index=False)
print(f'Saved: {CV_RESULTS_PATH}')

Saved: d:\My GitHub Projects\hotel-cancellation\reports\eval2\gradient_boosting_cv_results.csv


## 9. Freeze configurations and generate grouped OOF predictions

The selected General, City, and Resort parameter dictionaries are now frozen. OOF evidence is not used to trigger another search. For every fresh fold, the helper verifies zero group overlap, constructs a new estimator and complete preprocessing/model pipeline, fits only fold-training rows, and predicts only fold-validation rows. Coverage assertions guarantee exactly one prediction per row.

In [10]:
FROZEN_CONFIGURATIONS = {
    scope_name: dict(selected_estimator_parameters[scope_name])
    for scope_name in MODEL_SCOPES
}
assert set(FROZEN_CONFIGURATIONS) == set(MODEL_SCOPES)
display(pd.DataFrame([
    {'scope': scope_name, 'selected parameters': json_parameters(FROZEN_CONFIGURATIONS[scope_name])}
    for scope_name in MODEL_SCOPES
]))

def grouped_oof_predictions(scope, estimator_parameters):
    X_scope = scope['X']
    y_scope = scope['y']
    groups_scope = scope['groups']
    predictions = np.empty(len(X_scope), dtype=int)
    probabilities = np.empty(len(X_scope), dtype=float)
    validation_counts = np.zeros(len(X_scope), dtype=int)

    for fold_number, (train_pos, valid_pos) in enumerate(
        make_inner_cv().split(X_scope, y_scope, groups=groups_scope), start=1
    ):
        train_groups = set(groups_scope.iloc[train_pos])
        valid_groups = set(groups_scope.iloc[valid_pos])
        assert train_groups.isdisjoint(valid_groups), f'Group overlap in fold {fold_number}'
        assert np.all(validation_counts[valid_pos] == 0), f'Repeated OOF row in fold {fold_number}'

        fold_estimator = GradientBoostingClassifier(**dict(estimator_parameters))
        fold_pipeline = build_model_pipeline(X_scope.iloc[train_pos], fold_estimator)
        fold_pipeline.fit(X_scope.iloc[train_pos], y_scope.iloc[train_pos])
        predictions[valid_pos] = fold_pipeline.predict(X_scope.iloc[valid_pos])
        probabilities[valid_pos] = fold_pipeline.predict_proba(X_scope.iloc[valid_pos])[:, 1]
        validation_counts[valid_pos] += 1

    assert np.all(validation_counts == 1)
    assert validation_counts.sum() == len(X_scope)
    assert np.isfinite(probabilities).all()
    assert np.logical_and(probabilities >= 0, probabilities <= 1).all()
    assert set(np.unique(predictions)).issubset({0, 1})
    return pd.DataFrame(
        {'y_true': y_scope.to_numpy(), 'prediction': predictions, 'probability': probabilities},
        index=X_scope.index,
    )

oof_by_scope = {
    scope_name: grouped_oof_predictions(
        training_scopes[scope_name], FROZEN_CONFIGURATIONS[scope_name]
    )
    for scope_name in MODEL_SCOPES
}

,scope,selected parameters
0,General,"{""learning_rate"": 0.1, ""max_depth"": 3, ""max_fe..."
1,City,"{""learning_rate"": 0.1, ""max_depth"": 3, ""max_fe..."
2,Resort,"{""learning_rate"": 0.1, ""max_depth"": 3, ""max_fe..."


## 10. Training-only grouped OOF evaluation

ROC-AUC uses cancellation probabilities, while Accuracy, Precision, Recall, and F1 use predicted classes. The General OOF predictions are evaluated overall and on their City and Resort subsets; specialist OOF predictions are evaluated on their matching populations.

In [11]:
def oof_metric_record(model_scope, evaluation_population, oof_frame, mask=None):
    evaluated = oof_frame if mask is None else oof_frame.loc[mask]
    y_true = evaluated['y_true']
    y_pred = evaluated['prediction']
    y_prob = evaluated['probability']
    return {
        'algorithm': ALGORITHM,
        'scope': model_scope,
        'evaluation_population': evaluation_population,
        'stage': 'tuned_oof',
        'rows': len(evaluated),
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'recall': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        'f1': f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        'roc_auc': roc_auc_score(y_true, y_prob),
        'selected_parameters': json_parameters(FROZEN_CONFIGURATIONS[model_scope]),
    }

general_oof = oof_by_scope['General']
general_hotels = training_scopes['General']['X']['hotel']
city_mask = general_hotels.eq('City Hotel')
resort_mask = general_hotels.eq('Resort Hotel')

oof_records = [
    oof_metric_record('General', 'Overall', general_oof),
    oof_metric_record('General', 'City', general_oof, city_mask),
    oof_metric_record('General', 'Resort', general_oof, resort_mask),
    oof_metric_record('City', 'City', oof_by_scope['City']),
    oof_metric_record('Resort', 'Resort', oof_by_scope['Resort']),
]
oof_results = pd.DataFrame(oof_records)
assert len(oof_results) == 5
assert oof_results[['scope', 'evaluation_population']].apply(tuple, axis=1).tolist() == [
    ('General', 'Overall'), ('General', 'City'), ('General', 'Resort'),
    ('City', 'City'), ('Resort', 'Resort'),
]
assert oof_results['rows'].tolist() == [95_375, 63_044, 32_331, 63_044, 32_331]
display(oof_results)

,algorithm,scope,evaluation_population,stage,rows,accuracy,precision,recall,f1,roc_auc,selected_parameters
0,Gradient Boosting,General,Overall,tuned_oof,95375,0.777332,0.953279,0.420044,0.583139,0.896320,"{""learning_rate"": 0.1, ""max_depth"": 3, ""max_fe..."
1,Gradient Boosting,General,City,tuned_oof,63044,0.768606,0.959965,0.464351,0.625930,0.891017,"{""learning_rate"": 0.1, ""max_depth"": 3, ""max_fe..."
2,Gradient Boosting,General,Resort,tuned_oof,32331,0.794346,0.923640,0.291772,0.443459,0.900914,"{""learning_rate"": 0.1, ""max_depth"": 3, ""max_fe..."
3,Gradient Boosting,City,City,tuned_oof,63044,0.788656,0.949688,0.520659,0.672581,0.899619,"{""learning_rate"": 0.1, ""max_depth"": 3, ""max_fe..."
4,Gradient Boosting,Resort,Resort,tuned_oof,32331,0.810770,0.873392,0.381430,0.530972,0.907271,"{""learning_rate"": 0.1, ""max_depth"": 3, ""max_fe..."


## 11. Same-population specialist comparison

The novelty comparison holds the evaluation population constant. City-specific is compared with General-on-City, and Resort-specific with General-on-Resort. General Overall is not used as specialist evidence. Differences are training-only grouped OOF evidence and do not establish final superiority.

In [12]:
novelty_rows = []
for population, specialist_scope in (('City', 'City'), ('Resort', 'Resort')):
    general_row = oof_results.loc[
        oof_results['scope'].eq('General')
        & oof_results['evaluation_population'].eq(population)
    ].iloc[0]
    specialist_row = oof_results.loc[
        oof_results['scope'].eq(specialist_scope)
        & oof_results['evaluation_population'].eq(population)
    ].iloc[0]
    novelty_rows.append({
        'population': population,
        'comparison': f'{specialist_scope} specialist minus General-on-{population}',
        'F1 difference': specialist_row['f1'] - general_row['f1'],
        'Recall difference': specialist_row['recall'] - general_row['recall'],
        'ROC-AUC difference': specialist_row['roc_auc'] - general_row['roc_auc'],
    })
novelty_comparison = pd.DataFrame(novelty_rows)
display(novelty_comparison)
print('Interpretation: training-only grouped OOF evidence; final outer-test superiority is not claimed.')

,population,comparison,F1 difference,Recall difference,ROC-AUC difference
0,City,City specialist minus General-on-City,0.046651,0.056308,0.008602
1,Resort,Resort specialist minus General-on-Resort,0.087513,0.089657,0.006357


Interpretation: training-only grouped OOF evidence; final outer-test superiority is not claimed.


## 12. Save the five grouped OOF records

No model files are saved. Feature importance may be explored later for interpretation, but it is not used here to remove variables, select features, retune, or create another model variant.

In [13]:
OOF_RESULT_COLUMNS = [
    'algorithm', 'scope', 'evaluation_population', 'stage', 'rows',
    'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'selected_parameters',
]
oof_results = oof_results.loc[:, OOF_RESULT_COLUMNS]
assert len(oof_results) == 5
assert oof_results['stage'].eq('tuned_oof').all()
assert set(oof_results['scope']) == {'General', 'City', 'Resort'}
REPORT_DIR.mkdir(parents=True, exist_ok=True)
oof_results.to_csv(OOF_RESULTS_PATH, index=False)
print(f'Saved: {OOF_RESULTS_PATH}')

Saved: d:\My GitHub Projects\hotel-cancellation\reports\eval2\gradient_boosting_oof_results.csv


## Gradient Boosting Development Findings

The next cell produces a concise, result-dependent summary after local execution. Selected values are interpreted as configuration choices, not automatic proof of overfitting or superiority. Lower learning rate means a smaller contribution from each stage; more estimators means more sequential boosting stages; lower depth means simpler weak learners; a larger minimum leaf size means less-specific terminal regions; and `subsample < 1` means stochastic gradient boosting.

In [14]:
finding_lines = []
for row in comparison_rows:
    delta = row['F1 change']
    if delta < 0:
        tuning_text = 'the limited search did not improve F1'
    elif delta < 0.01:
        tuning_text = 'the F1 improvement was small'
    else:
        tuning_text = 'tuning improved grouped-CV F1'
    finding_lines.append(
        f"- **{row['scope']} baseline/tuning:** baseline F1={row['baseline F1']:.4f}; "
        f"tuned F1={row['tuned F1']:.4f}; {tuning_text}."
    )

for scope_name in MODEL_SCOPES:
    params = FROZEN_CONFIGURATIONS[scope_name]
    finding_lines.append(
        f"- **Selected {scope_name} configuration:** learning_rate={params['learning_rate']}, "
        f"n_estimators={params['n_estimators']}, max_depth={params['max_depth']}, "
        f"min_samples_leaf={params['min_samples_leaf']}, subsample={params['subsample']}, "
        f"max_features={params['max_features']}."
    )

for row in novelty_rows:
    finding_lines.append(
        f"- **{row['population']} same-population OOF:** specialist minus General "
        f"F1={row['F1 difference']:+.4f}, Recall={row['Recall difference']:+.4f}, "
        f"ROC-AUC={row['ROC-AUC difference']:+.4f}. This is training-only grouped OOF evidence."
    )

finding_lines.extend([
    '- **Strengths:** sequential shallow trees can capture nonlinear effects and interactions while providing cancellation probabilities.',
    '- **Limitations:** training is sequential, learning-rate/estimator-count choices interact, excessive complexity can overfit, and interpretation is less direct than for simpler models.',
    '- **Freeze decision:** the selected General, City, and Resort configurations are frozen for later final outer-test evaluation; OOF findings do not trigger retuning.',
])
display(Markdown('\n'.join(finding_lines)))

- **General baseline/tuning:** baseline F1=0.6991; tuned F1=0.5829; the limited search did not improve F1.
- **City baseline/tuning:** baseline F1=0.7519; tuned F1=0.6714; the limited search did not improve F1.
- **Resort baseline/tuning:** baseline F1=0.6706; tuned F1=0.5286; the limited search did not improve F1.
- **Selected General configuration:** learning_rate=0.1, n_estimators=50, max_depth=3, min_samples_leaf=10, subsample=1.0, max_features=sqrt.
- **Selected City configuration:** learning_rate=0.1, n_estimators=50, max_depth=3, min_samples_leaf=1, subsample=0.8, max_features=sqrt.
- **Selected Resort configuration:** learning_rate=0.1, n_estimators=50, max_depth=3, min_samples_leaf=1, subsample=0.8, max_features=sqrt.
- **City same-population OOF:** specialist minus General F1=+0.0467, Recall=+0.0563, ROC-AUC=+0.0086. This is training-only grouped OOF evidence.
- **Resort same-population OOF:** specialist minus General F1=+0.0875, Recall=+0.0897, ROC-AUC=+0.0064. This is training-only grouped OOF evidence.
- **Strengths:** sequential shallow trees can capture nonlinear effects and interactions while providing cancellation probabilities.
- **Limitations:** training is sequential, learning-rate/estimator-count choices interact, excessive complexity can overfit, and interpretation is less direct than for simpler models.
- **Freeze decision:** the selected General, City, and Resort configurations are frozen for later final outer-test evaluation; OOF findings do not trigger retuning.